## Lab Assignment 4: Topic Modeling with BERT
George Danut ASU ID: [redacted]

## Load Data

In [ ]:
## Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# File paths for each dataset

import os

In [ ]:
base_path = 'data/yelp_dataset_new'

review_file_path = os.path.join(
    base_path,
    'yelp_academic_dataset_review.json'
)

In [ ]:
## Load Yelp review data
reviews = pd.read_json(
    review_file_path)

reviews.head()

In [ ]:
## Review dataset structure
print(reviews.shape)

reviews.info()

## Calculate and visualize the distribution of review length.

In [ ]:
## Calculate review length
reviews['review_length'] = (
    reviews['text']
    .str.split()
    .str.len()
)

reviews[
    ['text', 'review_length']
].head()

In [ ]:
## Review length summary statistics
reviews['review_length'].describe()

In [ ]:
## Visualize review length distribution
plt.figure(figsize=(10, 6))

plt.hist(
    reviews['review_length'],
    bins=50,
    edgecolor='black'
)

plt.title('Distribution of Yelp Review Length')
plt.xlabel('Review Length (Words)')
plt.ylabel('Number of Reviews')

plt.show()

##Build a BERTopic Model with UMAP (target dimension of 5, n_neighbors of 15, and the usage of cosine similarity to measure the distance)

In [ ]:
## Install BERTopic
!pip install bertopic

In [ ]:
## Import BERTopic and UMAP
from bertopic import BERTopic
from umap import UMAP

In [ ]:
## Sample reviews for BERTopic
review_sample = reviews.sample(
    n=20000,
    random_state=42
).copy()

review_documents = (
    review_sample['text']
    .dropna()
    .astype(str)
    .tolist()
)

print('Number of Review Documents:', len(review_documents))

In [ ]:
## Create UMAP model
umap_model = UMAP(
    n_components=5,
    n_neighbors=15,
    metric='cosine',
    random_state=42
)

In [ ]:
## Create BERTopic model
topic_model = BERTopic(
    umap_model=umap_model,
    verbose=True
)

In [ ]:
## Fit BERTopic model
topics, probabilities = topic_model.fit_transform(
    review_documents
)

In [ ]:
## Display topic information
topic_model.get_topic_info()

## Visualize the most relevant words for the top 10 topics

In [ ]:
## Visualize top 10 words for top 10 topics
topic_model.visualize_barchart(
    top_n_topics=10,
    n_words=10
)

## Visualize the topic hierarchy for the top 30 topics

In [ ]:
## Visualize topic hierarchy for top 30 topics
topic_model.visualize_hierarchy(
    top_n_topics=30
)

## Reduce the number of topics to 15, visualize the most relevant words for each topic and the topic hierarchy

In [ ]:
## Reduce number of topics to 15
topic_model.reduce_topics(
    review_documents,
    nr_topics=15
)

In [ ]:
## Visualize most relevant words for reduced topics
topic_model.visualize_barchart(
    top_n_topics=15,
    n_words=10
)

In [ ]:
## Visualize hierarchy of reduced topics
topic_model.visualize_hierarchy(
    top_n_topics=15
)

## Get the topic allocation of the first 5 reviews in the data

In [ ]:
## Get topic allocation for each review
review_topic_info = topic_model.get_document_info(
    review_documents
)

## Display topic allocation for first 5 reviews
review_topic_info[
    ['Document', 'Topic', 'Name']
].head(5)

## Use get_topic() to show the most relevant words for the first topic

In [ ]:
## Show most relevant words for the first topic
topic_model.get_topic(0)

## Interpret each topic: What is the theme of each topic?

In [ ]:
## Display top 10 words for each meaningful topic
topic_numbers = topic_model.get_topic_info()['Topic']

for topic_number in topic_numbers:

    if topic_number != -1:

        topic_words = topic_model.get_topic(topic_number)

        words = [
            word
            for word, weight in topic_words[:10]
        ]

        print(
            f'Topic {topic_number}:',
            ', '.join(words)
        )

The reduced BERTopic model identified several distinct themes within Yelp customer reviews. Most topics correspond to recognizable business or service categories, including hotels, hair salons, automotive services, dental services, veterinary care, fitness facilities, wedding services, cleaning services, sporting venues, and specialty retail. Topic 0 was less interpretable because its highest-weight terms consisted primarily of common English words. Overall, the results indicate that BERTopic successfully separated reviews according to meaningful differences in the types of businesses and customer experiences being discussed.

## Visualize the topic frequency of the top 6 topics over time with the entire dataset

In [ ]:
## Convert sample review date to datetime
review_sample['date'] = pd.to_datetime(
    review_sample['date']
)

In [ ]:
## Calculate topic frequency over time
topics_over_time = topic_model.topics_over_time(
    review_documents,
    review_sample['date'].tolist(),
    topics=topic_model.topics_,
    nr_bins=20
)

topics_over_time.head()

In [ ]:
## Visualize top 6 topics over time
topic_model.visualize_topics_over_time(
    topics_over_time,
    top_n_topics=6
)

## Visualize the topics per star rating (e.g., 1, 2, 3, 4, 5)

In [ ]:
## Check star rating distribution
review_sample['stars'].value_counts().sort_index()

In [ ]:
## Calculate topics per star rating
topics_per_rating = topic_model.topics_per_class(
    review_documents,
    classes=review_sample['stars'].tolist()
)

topics_per_rating.head()

In [ ]:
## Visualize topics per star rating
topic_model.visualize_topics_per_class(
    topics_per_rating
)

## Comment on your findings based on the visualized topic frequency over time and topics per star rating in Code cells 10 and 11. In general, what are the major topics mentioned about good and bad restaurant experiences, respectively? What are the business implications?

The topic-frequency-over-time analysis shows that discussion across the major topics generally increased over the observed period, with the largest growth occurring after approximately 2010. Topic 0 was overwhelmingly the most frequent topic, although its representation consisted mainly of common stopwords and therefore did not provide a clearly interpretable theme. Other identifiable topics included hotels, salons, automotive services, dental services, and pet services.

The star-rating analysis shows that the major topics occurred most frequently in 4- and 5-star reviews, while substantially fewer topic occurrences were observed in 1- and 2-star reviews. This suggests that positive experiences make up a large portion of the sampled reviews; however, because the visualization presents raw frequencies, differences in the number of reviews at each rating level should also be considered. The BERTopic results do not clearly isolate restaurant-specific themes for good versus bad experiences because the sampled Yelp data includes multiple business categories. From a business perspective, the analysis demonstrates how topic modeling can be used to monitor which aspects of customer experiences are becoming more prominent over time and how those themes differ across satisfaction levels. Businesses could use this information to identify recurring customer concerns, reinforce themes associated with favorable ratings, and monitor changes in customer feedback over time.

## Specify a few topics in the restaurant reviews and use zero shot topic modeling to create a topic model

In [ ]:
## Specify restaurant topics
restaurant_topics = [
    'Food Quality',
    'Customer Service',
    'Wait Time',
    'Price and Value',
    'Atmosphere',
    'Cleanliness'
]

restaurant_topics

In [ ]:
## Create UMAP model for zero-shot topic modeling
zero_shot_umap = UMAP(
    n_components=5,
    n_neighbors=15,
    metric='cosine',
    random_state=42
)

In [ ]:
## Create zero-shot BERTopic model
zero_shot_model = BERTopic(
    zeroshot_topic_list=restaurant_topics,
    zeroshot_min_similarity=0.70,
    umap_model=zero_shot_umap,
    verbose=True
)

In [ ]:
## Fit zero-shot topic model
zero_shot_topics, zero_shot_probabilities = (
    zero_shot_model.fit_transform(
        review_documents
    )
)

In [ ]:
## Display zero-shot topic information
zero_shot_topic_info = (
    zero_shot_model.get_topic_info()
)

zero_shot_topic_info

In [ ]:
## Visualize zero-shot topics
zero_shot_model.visualize_barchart(
    top_n_topics=10,
    n_words=10
)

## Interpret each topic in the zero shot topic model: what is each topic about?

The zero-shot topic model focuses on six predefined dimensions of restaurant experiences: food quality, customer service, wait time, price and value, atmosphere, and cleanliness. Food Quality captures customers' evaluations of the meal itself, while Customer Service and Wait Time represent important aspects of service delivery. Price and Value reflects whether customers believe the experience justified its cost, while Atmosphere captures perceptions of the dining environment. Finally, Cleanliness represents customer perceptions of the physical condition and hygiene of the restaurant. Together, these topics represent several major dimensions that can influence the overall restaurant customer experience.

## Acknowledge if you have used any GenAI tools in this assignment and anyone you have worked together with on this assignment.

I worked on this assigment by myself. LLM was used to debug code and verify validity of outputs.

In [ ]:
## Convert notebook to HTML
!jupyter nbconvert \
    --to html \
    "data/LA4_Danut_George.ipynb"